In [1]:
import numpy as np, pandas as pd
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

df = pd.read_parquet("../data/processed/reviews_clean.parquet")
embeddings = np.load("../data/processed/embeddings.npy")
docs = df["text"].tolist()

umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0,
                  metric="cosine", random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=50, metric="euclidean",
                        cluster_selection_method="eom", prediction_data=True)
vectorizer = CountVectorizer(stop_words="english", min_df=10,
                             ngram_range=(1, 2))

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    verbose=True,
)
topics, probs = topic_model.fit_transform(docs, embeddings)

print(topic_model.get_topic_info().head(20))
print("Outliers:", (np.array(topics) == -1).mean())

2026-10-20 12:48:44,440 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-10-20 12:50:02,628 - BERTopic - Dimensionality - Completed ✓
2026-10-20 12:50:02,630 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-10-20 12:50:08,538 - BERTopic - Cluster - Completed ✓
2026-10-20 12:50:08,549 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-10-20 12:50:15,518 - BERTopic - Representation - Completed ✓


    Topic  Count                                   Name  \
0      -1  22623               -1_book_like_album_great   
1       0   4061                0_book_read_story_books   
2       1   3002             1_movie_film_movies_acting   
3       2   1178               2_dvd_bluray_disc_player   
4       3    880                 3_album_cd_voice_songs   
5       4    809                    4_toy_loves_son_old   
6       5    799             5_game_games_graphics_play   
7       6    607          6_item_product_order_received   
8       7    587       7_book_programming_examples_text   
9       8    546            8_bible_god_christian_jesus   
10      9    540                    9_pan_oven_pans_lid   
11     10    472            10_taste_tea_flavor_product   
12     11    463                 11_jazz_blues_album_cd   
13     12    446       12_condition_pages_book_received   
14     13    435             13_skin_hair_product_smell   
15     14    432      14_season_episodes_series_seasons 

In [2]:
topic_model.save("../models/bertopic_v1", serialization="safetensors",
                 save_ctfidf=True,
                 save_embedding_model="all-MiniLM-L6-v2")
df["topic_v1"] = topics
df[["review_id", "topic_v1"]].to_parquet(
    "../data/processed/topics_v1.parquet", index=False)

In [ ]:
import sys
sys.path.append("..")
from src.topics import reduce_outliers_embeddings, sample_reassigned_reviews, get_default_vectorizer

# Load fresh copy of bertopic_v1 (Rule 5)
topic_model_v1 = BERTopic.load("../models/bertopic_v1")
orig_topics = pd.read_parquet("../data/processed/topics_v1.parquet")["topic_v1"].values

# Reduce outliers using embeddings
vectorizer = get_default_vectorizer()
topic_model_v1, new_topics = reduce_outliers_embeddings(
    topic_model=topic_model_v1,
    docs=docs,
    topics=orig_topics,
    embeddings=embeddings,
    vectorizer=vectorizer,
)
print("Outlier fraction after reduction:", (np.array(new_topics) == -1).mean())

# Save reduced assignments to topics_v1_reduced.parquet
df["topic_v1_reduced"] = new_topics
df[["review_id", "topic_v1_reduced"]].to_parquet(
    "../data/processed/topics_v1_reduced.parquet", index=False
)

In [ ]:
samples = sample_reassigned_reviews(
    df=df,
    orig_topics=orig_topics,
    new_topics=new_topics,
    topic_model=topic_model_v1,
    n=8,
    random_state=1,
)

for _, row in samples.iterrows():
    print(f"Review {row['review_id']} -> {row['new_topic_name']}:")
    print(row['text_200'])
    print("-" * 70)

In [5]:
# === Phase 3: Looser clustering v2 ===
from src.topics import fit_bertopic_v2, count_topics

topic_model_v2, topics_v2, probs_v2 = fit_bertopic_v2(
    docs=docs,
    embeddings=embeddings,
    random_state=42,
)

print("BERTopic v2 Topic Count (excluding -1):", count_topics(topic_model_v2))
print("BERTopic v2 Outlier Fraction:", (np.array(topics_v2) == -1).mean())

# Save model and assignments
topic_model_v2.save(
    "../models/bertopic_v2",
    serialization="safetensors",
    save_ctfidf=True,
    save_embedding_model="all-MiniLM-L6-v2",
)

df["topic_v2"] = topics_v2
df[["review_id", "topic_v2"]].to_parquet(
    "../data/processed/topics_v2.parquet", index=False
)

In [6]:
import pandas as pd
t = pd.read_parquet("../data/processed/topics_v1.parquet")
print((t["topic_v1"] == -1).mean())          # expect ~0.45
print(t["topic_v1"].nunique() - 1)           # expect 108

0.45246
109


In [7]:
t = pd.read_parquet("../data/processed/topics_v1.parquet")
real = sorted(set(t["topic_v1"]) - {-1})
print(len(real), min(real), max(real))   # expect 109, 0, 108
print(t["topic_v1"].isna().sum())        # expect 0


109 0 108
0


In [ ]:
from src.topics import count_topics
print("Topic count:", count_topics(topic_model_v1))

In [ ]:
# === Phase 2: Merge to a dashboard-sized number ===
from src.topics import merge_topics, get_topic_summary, count_topics

# Load fresh copy of bertopic_v1 (Rule 5)
topic_model_merged = BERTopic.load("../models/bertopic_v1")
topic_model_merged, merged_topics = merge_topics(topic_model_merged, docs, nr_topics=30)

print("Topic count after merge:", count_topics(topic_model_merged))

# Save merged model and parquet
topic_model_merged.save(
    "../models/bertopic_v1_merged",
    serialization="safetensors",
    save_ctfidf=True,
    save_embedding_model="all-MiniLM-L6-v2",
)

df["topic_v1_merged"] = merged_topics
df[["review_id", "topic_v1_merged"]].to_parquet(
    "../data/processed/topics_v1_merged.parquet", index=False
)

summary_merged = get_topic_summary(topic_model_merged, exclude_outliers=False)
print(summary_merged.to_string(index=False))

In [ ]:
# === Phase 4: Fair NPMI comparison (LDA vs BERTopic) ===
import os, sys
if '..' not in sys.path:
    sys.path.insert(0, '..')
import pandas as pd
from src.topics import (
    build_gensim_dictionary,
    compute_coherence_lda,
    compute_coherence_bertopic,
)

tokens_df = pd.read_parquet("../data/processed/tokens.parquet")
texts = tokens_df["tokens"].tolist()
docs = df["text"].tolist()

dictionary = build_gensim_dictionary(texts, no_below=10, no_above=0.4)
print("Dictionary size:", len(dictionary))

rows = []

# 1. LDA (k=10)
s_lda10 = compute_coherence_lda(
    "../models/lda_k10.model", texts=texts, dictionary=dictionary, processes=1
)
rows.append({
    "Method": "LDA (k=10)",
    "Topics": 10,
    "Outliers": "0.0%",
    "NPMI": round(s_lda10, 4),
})

# 2. LDA (k=15)
s_lda15 = compute_coherence_lda(
    "../models/lda_k15.model", texts=texts, dictionary=dictionary, processes=1
)
rows.append({
    "Method": "LDA (k=15)",
    "Topics": 15,
    "Outliers": "0.0%",
    "NPMI": round(s_lda15, 4),
})

# 3. BERTopic v1 (original topics with outliers)
t_v1 = pd.read_parquet("../data/processed/topics_v1.parquet")["topic_v1"].values
s_v1, n_v1, out_v1 = compute_coherence_bertopic(
    "../models/bertopic_v1",
    docs=docs,
    topics=t_v1,
    texts=texts,
    dictionary=dictionary,
    processes=1,
)
rows.append({
    "Method": "BERTopic v1",
    "Topics": n_v1,
    "Outliers": f"{out_v1 * 100:.1f}%",
    "NPMI": round(s_v1, 4),
})

# 4. BERTopic v2 (original topics with outliers)
t_v2 = pd.read_parquet("../data/processed/topics_v2.parquet")["topic_v2"].values
s_v2, n_v2, out_v2 = compute_coherence_bertopic(
    "../models/bertopic_v2",
    docs=docs,
    topics=t_v2,
    texts=texts,
    dictionary=dictionary,
    processes=1,
)
rows.append({
    "Method": "BERTopic v2",
    "Topics": n_v2,
    "Outliers": f"{out_v2 * 100:.1f}%",
    "NPMI": round(s_v2, 4),
})

# 5. BERTopic v1 merged (29 themes)
t_vm = pd.read_parquet("../data/processed/topics_v1_merged.parquet")["topic_v1_merged"].values
s_vm, n_vm, out_vm = compute_coherence_bertopic(
    "../models/bertopic_v1_merged",
    docs=docs,
    topics=t_vm,
    texts=texts,
    dictionary=dictionary,
    processes=1,
)
rows.append({
    "Method": "BERTopic v1 (merged 29 themes)",
    "Topics": n_vm,
    "Outliers": f"{out_vm * 100:.1f}%",
    "NPMI": round(s_vm, 4),
})

df_comp = pd.DataFrame(rows)
os.makedirs("../results", exist_ok=True)
df_comp.to_csv("../results/topic_comparison.csv", index=False)
with open("../results/topic_comparison.csv", "a", encoding="utf-8") as f:
    f.write("# Note: Coherence depends on topic count and outlier handling; BERTopic NPMI is computed on clustered topics only, excluding the outlier group.\n")
print(df_comp.to_string(index=False))
print("\nNote: Coherence depends on topic count and outlier handling; BERTopic NPMI is computed on clustered topics only, excluding the outlier group.")


In [ ]:
# === Phase 6: LLM theme labels (v2 objective prompt) ===
import os, sys, json
if '..' not in sys.path:
    sys.path.insert(0, '..')
import numpy as np, pandas as pd

v1_path = "../data/processed/theme_labels.json"
v2_path = "../data/processed/theme_labels_v2.json"

with open(v1_path, "r", encoding="utf-8") as f:
    v1_labels = json.load(f)

with open(v2_path, "r", encoding="utf-8") as f:
    v2_labels = json.load(f)

table_rows = []
for tid in sorted(int(k) for k in v2_labels.keys()):
    k = str(tid)
    table_rows.append({
        "Topic ID": tid,
        "BERTopic Name": v2_labels[k]["bertopic_name"],
        "Old Label": v1_labels.get(k, {}).get("label", "N/A"),
        "New Label": v2_labels[k]["label"],
        "Count": v2_labels[k]["count"],
    })

df_comparison = pd.DataFrame(table_rows)
print(df_comparison.to_string(index=False))
